# ETF 动量轮动：负动量时切换到防守资产或空仓

这个 notebook 在现有 ETF 动量数据项目基础上，加入一个更稳健的风险控制规则：

> 每月末计算行业/主题 ETF 的过去 6 个月动量。若最强行业 ETF 动量为正，则下月持有该 ETF；若最强行业 ETF 动量为负或不可用，则不追行业，改为持有防守资产；若防守资产也没有正动量，则空仓。

注意：这是学习和研究用样例，不构成投资建议。真实交易还要考虑滑点、最小佣金、成交约束、跟踪误差和税费等问题。

## 1. 规则说明

本 notebook 使用以下默认口径：

- 数据文件：`../data/etf_momentum_daily_eastmoney_qfq.csv`
- 价格：前复权收盘价 `close`
- 行业轮动池：`bucket == 'sector'`
- 防守资产池：`bucket == 'defensive'`，当前已成功下载的是 `511010 国债ETF`、`511260 十年国债ETF`
- 调仓频率：月末生成信号，下一个交易日持仓，避免未来函数
- 动量窗口：126 个交易日，约 6 个月
- 交易成本：每次换仓按组合净值扣 `0.02%`，只是简化估计
- 规则：
  1. 在每个调仓日，找过去 126 日收益最高的行业 ETF；
  2. 若它的动量 `> 0`，下期持有它；
  3. 若它的动量 `<= 0`，在防守资产里找过去 126 日收益最高者；
  4. 若最佳防守资产动量 `> 0`，下期持有该防守资产；
  5. 否则空仓，日收益记为 0。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
DATA_PATH = BASE / "data" / "etf_momentum_daily_eastmoney_qfq.csv"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

LOOKBACK_DAYS = 126
FEE_RATE = 0.0002
INITIAL_NAV = 1.0

DATA_PATH

WindowsPath('d:/Quant/data/etf_momentum_daily_eastmoney_qfq.csv')

## 2. 读取数据并构建价格宽表

In [2]:
df = pd.read_csv(DATA_PATH, dtype={"symbol": "string"}, parse_dates=["date"])
df["symbol"] = df["symbol"].astype("string").str.strip()
df["close"] = pd.to_numeric(df["close"], errors="coerce")
df = df.sort_values(["symbol", "date"])

meta = (
    df.groupby("symbol")
    .agg(
        name=("name", "last"),
        bucket=("bucket", "last"),
        theme=("theme", "last"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        rows=("date", "size"),
    )
    .reset_index()
)

close = df.pivot(index="date", columns="symbol", values="close").sort_index()
daily_ret = close.pct_change(fill_method=None).fillna(0.0)

sector_symbols = meta.loc[meta["bucket"].eq("sector"), "symbol"].tolist()
defensive_symbols = meta.loc[meta["bucket"].eq("defensive"), "symbol"].tolist()
benchmark_symbols = meta.loc[meta["bucket"].eq("benchmark"), "symbol"].tolist()

print(f"数据区间: {close.index.min().date()} ~ {close.index.max().date()}")
print(f"ETF 数量: {close.shape[1]}")
print(f"行业/主题 ETF: {len(sector_symbols)}", sector_symbols)
print(f"防守资产: {len(defensive_symbols)}", defensive_symbols)
print(f"基准资产: {len(benchmark_symbols)}", benchmark_symbols)
meta.sort_values(["bucket", "symbol"])

数据区间: 2015-01-05 ~ 2026-06-18
ETF 数量: 18
行业/主题 ETF: 14 ['159928', '512010', '512170', '512400', '512480', '512660', '512690', '512800', '512880', '512980', '515030', '515220', '515230', '515790']
防守资产: 2 ['511010', '511260']
基准资产: 2 ['159915', '510300']


,symbol,name,bucket,theme,first_date,last_date,rows
0,159915,创业板ETF易方达,benchmark,创业板,2015-01-05,2026-06-18,2782
2,510300,沪深300ETF华泰柏瑞,benchmark,沪深300,2015-01-05,2026-06-18,2783
3,511010,国债ETF国泰,defensive,国债,2015-01-05,2026-06-18,2783
4,511260,十年国债ETF国泰,defensive,十年国债,2017-08-24,2026-06-18,2137
1,159928,消费ETF汇添富,sector,消费,2015-01-05,2026-06-18,2783
5,512010,医药ETF易方达,sector,医药,2015-01-05,2026-06-18,2781
6,512170,医疗ETF华宝,sector,医疗,2019-06-17,2026-06-18,1699
7,512400,有色金属ETF南方,sector,有色金属,2017-09-01,2026-06-18,2132
8,512480,半导体ETF国联安,sector,半导体,2019-06-12,2026-06-18,1702
9,512660,军工ETF国泰,sector,军工,2016-08-08,2026-06-18,2393


## 3. 计算月末动量与下期持仓

In [3]:
def month_end_trading_dates(index: pd.DatetimeIndex) -> pd.DatetimeIndex:
    return pd.Series(index, index=index).groupby(index.to_period("M")).last().values


rebalance_dates = pd.DatetimeIndex(month_end_trading_dates(close.index))
momentum = close / close.shift(LOOKBACK_DAYS) - 1.0

decisions = []
for signal_date in rebalance_dates:
    if signal_date not in momentum.index:
        continue
    row = momentum.loc[signal_date]
    sector_mom = row[sector_symbols].dropna().sort_values(ascending=False)
    defensive_mom = row[defensive_symbols].dropna().sort_values(ascending=False)

    chosen_symbol = "CASH"
    chosen_bucket = "cash"
    chosen_name = "空仓"
    chosen_theme = "现金"
    reason = "行业动量不可用，且防守资产动量不可用，空仓"
    selected_momentum = np.nan
    best_sector_symbol = None
    best_sector_momentum = np.nan
    best_defensive_symbol = None
    best_defensive_momentum = np.nan

    if not sector_mom.empty:
        best_sector_symbol = sector_mom.index[0]
        best_sector_momentum = float(sector_mom.iloc[0])

    if not defensive_mom.empty:
        best_defensive_symbol = defensive_mom.index[0]
        best_defensive_momentum = float(defensive_mom.iloc[0])

    if pd.notna(best_sector_momentum) and best_sector_momentum > 0:
        chosen_symbol = best_sector_symbol
        chosen_bucket = "sector"
        selected_momentum = best_sector_momentum
        reason = "最佳行业/主题 ETF 动量为正，持有该 ETF"
    elif pd.notna(best_defensive_momentum) and best_defensive_momentum > 0:
        chosen_symbol = best_defensive_symbol
        chosen_bucket = "defensive"
        selected_momentum = best_defensive_momentum
        reason = "最佳行业/主题 ETF 动量为负或不可用，切换到正动量防守资产"
    else:
        selected_momentum = 0.0
        reason = "最佳行业/主题 ETF 动量为负或不可用，防守资产也无正动量，空仓"

    if chosen_symbol != "CASH":
        meta_row = meta.set_index("symbol").loc[chosen_symbol]
        chosen_name = meta_row["name"]
        chosen_theme = meta_row["theme"]

    decisions.append(
        {
            "signal_date": signal_date,
            "chosen_symbol": chosen_symbol,
            "chosen_name": chosen_name,
            "chosen_bucket": chosen_bucket,
            "chosen_theme": chosen_theme,
            "selected_momentum": selected_momentum,
            "best_sector_symbol": best_sector_symbol,
            "best_sector_momentum": best_sector_momentum,
            "best_defensive_symbol": best_defensive_symbol,
            "best_defensive_momentum": best_defensive_momentum,
            "reason": reason,
        }
    )

decisions = pd.DataFrame(decisions)
decisions.head(), decisions.tail()

(  signal_date chosen_symbol chosen_name chosen_bucket chosen_theme  \
 0  2015-01-30          CASH          空仓          cash           现金   
 1  2015-02-27          CASH          空仓          cash           现金   
 2  2015-03-31          CASH          空仓          cash           现金   
 3  2015-04-30          CASH          空仓          cash           现金   
 4  2015-05-29          CASH          空仓          cash           现金   
 
    selected_momentum best_sector_symbol  best_sector_momentum  \
 0                0.0                NaN                   NaN   
 1                0.0                NaN                   NaN   
 2                0.0                NaN                   NaN   
 3                0.0                NaN                   NaN   
 4                0.0                NaN                   NaN   
 
   best_defensive_symbol  best_defensive_momentum  \
 0                   NaN                      NaN   
 1                   NaN                      NaN   
 2             

## 4. 生成每日持仓，计算策略净值

In [4]:
positions = pd.Series("CASH", index=close.index, name="position", dtype="object")
decision_by_date = decisions.set_index("signal_date")

current_position = "CASH"
for date in close.index:
    positions.loc[date] = current_position
    if date in decision_by_date.index:
        # 月末收盘后才能知道信号，因此从下一个交易日开始生效。
        current_position = decision_by_date.loc[date, "chosen_symbol"]

strategy_ret = pd.Series(0.0, index=close.index, name="strategy_return")
for symbol in close.columns:
    mask = positions.eq(symbol)
    strategy_ret.loc[mask] = daily_ret.loc[mask, symbol]

# 换仓当天扣一次简化成本。现金和 ETF 之间切换也视为交易。
trades = positions.ne(positions.shift(1)).fillna(False)
trades.iloc[0] = False
strategy_ret_after_cost = strategy_ret.copy()
strategy_ret_after_cost.loc[trades] -= FEE_RATE

strategy_nav = (1.0 + strategy_ret_after_cost).cumprod() * INITIAL_NAV
result_daily = pd.DataFrame(
    {
        "date": close.index,
        "position": positions.values,
        "strategy_return": strategy_ret.values,
        "strategy_return_after_cost": strategy_ret_after_cost.values,
        "trade": trades.values,
        "nav": strategy_nav.values,
    }
)

result_daily.tail()

,date,position,strategy_return,strategy_return_after_cost,trade,nav
2778,2026-06-12,512480,-0.011268,-0.011268,False,7.969149
2779,2026-06-15,512480,0.059354,0.059354,False,8.442152
2780,2026-06-16,512480,0.010309,0.010309,False,8.529184
2781,2026-06-17,512480,0.059006,0.059006,False,9.032459
2782,2026-06-18,512480,0.039380,0.039380,False,9.388157


## 5. 绩效指标与基准对比

In [5]:
def max_drawdown(nav: pd.Series) -> float:
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return float(dd.min())


def annualized_return(nav: pd.Series, periods_per_year: int = 252) -> float:
    if len(nav) < 2:
        return np.nan
    total = nav.iloc[-1] / nav.iloc[0] - 1.0
    years = len(nav) / periods_per_year
    return float((1.0 + total) ** (1.0 / years) - 1.0)


def annualized_volatility(ret: pd.Series, periods_per_year: int = 252) -> float:
    return float(ret.std() * np.sqrt(periods_per_year))


def sharpe_like(ret: pd.Series, periods_per_year: int = 252) -> float:
    vol = annualized_volatility(ret, periods_per_year)
    if vol == 0 or pd.isna(vol):
        return np.nan
    return float(ret.mean() * periods_per_year / vol)


benchmark_symbol = "510300" if "510300" in close.columns else benchmark_symbols[0]
benchmark_nav = (1.0 + daily_ret[benchmark_symbol]).cumprod()

metrics = pd.DataFrame(
    [
        {
            "name": "momentum_defensive_or_cash",
            "total_return": strategy_nav.iloc[-1] / strategy_nav.iloc[0] - 1.0,
            "annualized_return": annualized_return(strategy_nav),
            "annualized_volatility": annualized_volatility(strategy_ret_after_cost),
            "max_drawdown": max_drawdown(strategy_nav),
            "sharpe_like_no_rf": sharpe_like(strategy_ret_after_cost),
            "trade_count": int(trades.sum()),
            "cash_days": int(positions.eq("CASH").sum()),
            "defensive_days": int(positions.isin(defensive_symbols).sum()),
            "sector_days": int(positions.isin(sector_symbols).sum()),
        },
        {
            "name": f"buy_hold_{benchmark_symbol}",
            "total_return": benchmark_nav.iloc[-1] / benchmark_nav.iloc[0] - 1.0,
            "annualized_return": annualized_return(benchmark_nav),
            "annualized_volatility": annualized_volatility(daily_ret[benchmark_symbol]),
            "max_drawdown": max_drawdown(benchmark_nav),
            "sharpe_like_no_rf": sharpe_like(daily_ret[benchmark_symbol]),
            "trade_count": 0,
            "cash_days": 0,
            "defensive_days": 0,
            "sector_days": len(benchmark_nav),
        },
    ]
)

metrics

,name,total_return,annualized_return,annualized_volatility,max_drawdown,sharpe_like_no_rf,trade_count,cash_days,defensive_days,sector_days
0,momentum_defensive_or_cash,8.388157,0.224805,0.361407,-0.479012,0.74247,46,142,239,2402
1,buy_hold_510300,0.725164,0.050618,0.265148,-0.529723,0.31977,0,0,0,2783


## 6. 查看调仓记录

In [6]:
decisions_display = decisions.copy()
for column in ["selected_momentum", "best_sector_momentum", "best_defensive_momentum"]:
    decisions_display[column] = decisions_display[column].map(lambda x: f"{x:.2%}" if pd.notna(x) else "")
decisions_display.tail(20)

,signal_date,chosen_symbol,chosen_name,chosen_bucket,chosen_theme,selected_momentum,best_sector_symbol,best_sector_momentum,best_defensive_symbol,best_defensive_momentum,reason
118,2024-11-29,512480,半导体ETF国联安,sector,半导体,47.58%,512480,47.58%,511260,3.68%,最佳行业/主题 ETF 动量为正，持有该 ETF
119,2024-12-31,512880,证券ETF国泰,sector,证券,46.07%,512880,46.07%,511260,5.21%,最佳行业/主题 ETF 动量为正，持有该 ETF
120,2025-01-27,515230,软件ETF国泰,sector,软件,49.33%,515230,49.33%,511260,5.37%,最佳行业/主题 ETF 动量为正，持有该 ETF
121,2025-02-28,515230,软件ETF国泰,sector,软件,75.10%,515230,75.10%,511260,4.11%,最佳行业/主题 ETF 动量为正，持有该 ETF
122,2025-03-31,515230,软件ETF国泰,sector,软件,68.81%,515230,68.81%,511260,2.27%,最佳行业/主题 ETF 动量为正，持有该 ETF
123,2025-04-30,512480,半导体ETF国联安,sector,半导体,8.38%,512480,8.38%,511260,4.62%,最佳行业/主题 ETF 动量为正，持有该 ETF
124,2025-05-30,512800,银行ETF华宝,sector,银行,15.43%,512800,15.43%,511260,3.57%,最佳行业/主题 ETF 动量为正，持有该 ETF
125,2025-06-30,512800,银行ETF华宝,sector,银行,18.40%,512800,18.40%,511260,1.38%,最佳行业/主题 ETF 动量为正，持有该 ETF
126,2025-07-31,512400,有色金属ETF南方,sector,有色金属,20.68%,512400,20.68%,511260,0.14%,最佳行业/主题 ETF 动量为正，持有该 ETF
127,2025-08-29,512400,有色金属ETF南方,sector,有色金属,46.66%,512400,46.66%,511010,0.68%,最佳行业/主题 ETF 动量为正，持有该 ETF


## 7. 保存结果文件

In [7]:
daily_path = OUTPUT_DIR / "etf_momentum_defensive_or_cash_daily.csv"
decision_path = OUTPUT_DIR / "etf_momentum_defensive_or_cash_decisions.csv"
metrics_path = OUTPUT_DIR / "etf_momentum_defensive_or_cash_metrics.csv"

result_daily.to_csv(daily_path, index=False, encoding="utf-8-sig")
decisions.to_csv(decision_path, index=False, encoding="utf-8-sig")
metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")

print(f"已保存每日净值: {daily_path}")
print(f"已保存调仓决策: {decision_path}")
print(f"已保存绩效指标: {metrics_path}")

已保存每日净值: d:\Quant\outputs\etf_momentum_defensive_or_cash_daily.csv
已保存调仓决策: d:\Quant\outputs\etf_momentum_defensive_or_cash_decisions.csv
已保存绩效指标: d:\Quant\outputs\etf_momentum_defensive_or_cash_metrics.csv


## 8. 如何解读这个策略

这个规则的重点不是“预测市场”，而是减少一个常见错误：

> 当所有行业 ETF 都是负动量时，还强行选一个“跌得最少的行业”满仓。

加入防守资产/空仓规则后，策略会承认“没有好机会”这种状态。  
但它也有明显代价：

- 可能错过 V 型反转初期；
- 防守资产本身也有利率风险或跟踪误差；
- 空仓规则可能降低长期暴露，牛市中容易跑输；
- 6 个月动量窗口只是教学默认值，不是最优参数。

后续可以继续做参数稳健性测试：3/6/9/12 个月动量、不同防守资产、是否允许防守资产负动量仍持有、不同交易成本等。